# Time Series Anomaly Detection

This notebook demonstrates quantum anomaly detection on time series data using synthetic
sensor signals. We compare four quantum methods against classical baselines.

**Anomaly types**: spike anomalies, frequency shifts, and level shifts injected into
sinusoidal signals.

**Quantum methods**: Quantum Kernel + One-Class SVM, VQC Autoencoder, QAOA Regression Residuals,
Quantum Distance k-NN.

**Classical baselines**: Isolation Forest, One-Class SVM, Local Outlier Factor, Elliptic Envelope.

In [ ]:
# ============================================================
# Configuration — change these values to experiment
# ============================================================
SEED = 42
N_QUBITS = 6
N_SAMPLES = 300
WINDOW_SIZE = 64
CONTAMINATION = 0.05
N_KERNEL_SAMPLES = 100
N_QAOA_POINTS = 12
CIRCUIT_SCALE = 0.6  # Scale factor for circuit drawings


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

## 1. Data Loading & Exploration

We generate synthetic time series windows: normal windows are sinusoidal signals with
slight frequency and phase variation. Anomalous windows contain spikes, frequency shifts,
or level shifts.

In [ ]:
from quantum_anomaly_detection.data.time_series import (
    load_synthetic_timeseries, extract_window_features, preprocess_timeseries
)

X_raw, y = load_synthetic_timeseries(
    n_samples=N_SAMPLES, window_size=WINDOW_SIZE,
    anomaly_fraction=CONTAMINATION, seed=SEED
)
print(f'Dataset: {X_raw.shape[0]} windows of size {X_raw.shape[1]}')
print(f'Normal: {(y==0).sum()}, Anomaly: {(y==1).sum()}')

In [ ]:
# Plot sample windows
fig, axes = plt.subplots(2, 4, figsize=(14, 5))
normal_idx = np.where(y == 0)[0][:4]
anomaly_idx = np.where(y == 1)[0][:4]

for i, idx in enumerate(normal_idx):
    axes[0, i].plot(X_raw[idx], color='steelblue', linewidth=0.8)
    axes[0, i].set_title(f'Normal #{i+1}', fontsize=9)
for i, idx in enumerate(anomaly_idx):
    axes[1, i].plot(X_raw[idx], color='red', linewidth=0.8)
    axes[1, i].set_title(f'Anomaly #{i+1}', fontsize=9)

fig.suptitle('Sample Time Series Windows')
plt.tight_layout()
plt.show()

## 2. Preprocessing

We extract statistical features from each window (mean, std, max, min, skewness,
kurtosis, top FFT magnitudes), then apply PCA and scale to [0, pi] for quantum encoding.

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=SEED, stratify=y
)

# Preprocess: extract features + PCA + scale to [0, pi]
X_train = preprocess_timeseries(X_train_raw, n_components=N_QUBITS, fit_data=X_train_raw)
X_test = preprocess_timeseries(X_test_raw, n_components=N_QUBITS, fit_data=X_train_raw)

# Use only normal training data for one-class methods
X_train_normal = X_train[y_train == 0]
print(f'Train: {len(X_train)} ({(y_train==0).sum()} normal), Test: {len(X_test)}')
print(f'Feature shape: {X_train.shape[1]} components, range [{X_train.min():.2f}, {X_train.max():.2f}]')

## 3. Quantum Kernel + One-Class SVM

The quantum kernel method encodes each feature vector into a quantum state using a
parameterized feature map. The kernel value K(x_i, x_j) = |<phi(x_i)|phi(x_j)>|^2
measures similarity in quantum Hilbert space. This kernel matrix is then used with a
One-Class SVM for novelty detection.

For time series, the features extracted from each window (statistical + frequency domain)
capture the signal characteristics that distinguish normal from anomalous behavior.

In [ ]:
from quantum_anomaly_detection.circuits.feature_maps import build_zz_feature_map
from quantum_anomaly_detection.methods.quantum_kernel import quantum_kernel_svm, compute_kernel_matrix
from quantum_anomaly_detection.visualization.plots import plot_kernel_matrix

# Build and visualize the feature map
feature_map_zz = build_zz_feature_map(N_QUBITS, reps=1)
feature_map_zz.draw('mpl', fold=20, scale=CIRCUIT_SCALE)

In [ ]:
# Compute kernel matrix on subset and run One-Class SVM
rng = np.random.default_rng(SEED)
n_sub = min(N_KERNEL_SAMPLES, len(X_train_normal))
idx_train = rng.choice(len(X_train_normal), n_sub, replace=False)
idx_test = rng.choice(len(X_test), min(N_KERNEL_SAMPLES, len(X_test)), replace=False)

preds_qk, scores_qk, _ = quantum_kernel_svm(
    X_train_normal[idx_train], X_test[idx_test], feature_map_zz, nu=CONTAMINATION
)

# Visualize kernel matrix
K = compute_kernel_matrix(X_train_normal[idx_train[:30]], feature_map_zz)
plot_kernel_matrix(K, title='Quantum Kernel Matrix (ZZ Feature Map)')
plt.show()

In [ ]:
from quantum_anomaly_detection.evaluation.metrics import compute_metrics

results = {}
m = compute_metrics(y_test[idx_test], preds_qk, scores_qk)
results['Quantum Kernel SVM'] = m
print('Quantum Kernel One-Class SVM:', {k: f'{v:.3f}' for k, v in m.items()})

## 4. VQC Autoencoder

The Variational Quantum Circuit Autoencoder compresses n qubits into a smaller latent
space. Using the "trash qubit" approach, the reconstruction quality is measured by how
close the non-latent (trash) qubits are to |0>. Normal data should compress well
(low loss), while anomalies yield high reconstruction error.

In [ ]:
from quantum_anomaly_detection.circuits.autoencoder import build_autoencoder_compact

n_latent = N_QUBITS // 2
ae_compact, ae_encoder, ae_decoder = build_autoencoder_compact(
    N_QUBITS, n_latent, encoder_reps=1, decoder_reps=1
)
print(f'Autoencoder: {N_QUBITS} qubits -> {n_latent} latent')
print('\nCompact view:')
display(ae_compact.draw('mpl', scale=CIRCUIT_SCALE))
print('\nEncoder detail:')
display(ae_encoder.draw('mpl', fold=20, scale=CIRCUIT_SCALE))


In [ ]:
# Train on normal data (small subset for speed)
train_sub = X_train_normal[:40]
params_ae, circuit_ae, history_ae = train_autoencoder(
    train_sub, n_latent=n_latent, encoder_reps=1, decoder_reps=1,
    maxiter=100, seed=SEED
)

plot_optimization_history(history_ae, title='VQC Autoencoder Training')
plt.show()

In [ ]:
# Score test data
scores_ae = score_anomalies(X_test, params_ae, circuit_ae, n_latent)
preds_ae = detect_anomalies(scores_ae, contamination=CONTAMINATION)

m = compute_metrics(y_test, preds_ae, scores_ae)
results['VQC Autoencoder'] = m
print('VQC Autoencoder:', {k: f'{v:.3f}' for k, v in m.items()})

## 5. QAOA Regression Residuals

This method combines classical regression with quantum optimization. We first fit a
linear model to predict feature values, then compute residuals. Points with large
residuals are candidate anomalies. QAOA solves a binary optimization problem to
determine the optimal anomaly/normal labeling.

The QUBO formulation encodes: prefer labeling high-residual points as anomalies,
with a smoothness penalty to avoid isolated labels.

In [ ]:
from quantum_anomaly_detection.methods.qaoa_regression import (
    fit_regression, run_qaoa_thresholding
)
from quantum_anomaly_detection.circuits.qaoa import build_thresholding_hamiltonian, build_qaoa_circuit

# Extract features for regression (use raw features, predict next from previous)
features_all = np.array([extract_window_features(w) for w in X_test_raw])

# Fit regression: predict feature vector from itself (reconstruction-based)
model_reg, residuals = fit_regression(features_all[:, :4], features_all[:, 4:])
print(f'Residuals: min={residuals.min():.3f}, max={residuals.max():.3f}, mean={residuals.mean():.3f}')

In [ ]:
# Subsample for QAOA (limited qubits)
idx_qaoa = rng.choice(len(residuals), N_QAOA_POINTS, replace=False)
res_sub = residuals[idx_qaoa]
y_sub = y_test[idx_qaoa]

# Visualize the QAOA circuit
H_thresh = build_thresholding_hamiltonian(res_sub / res_sub.max(), penalty=0.5)
qaoa_circ = build_qaoa_circuit(H_thresh, reps=2)
print(f'QAOA circuit: {qaoa_circ.num_qubits} qubits, {qaoa_circ.num_parameters} parameters')

In [ ]:
# Run QAOA thresholding
labels_qaoa_reg, history_qaoa_reg = run_qaoa_thresholding(
    res_sub, penalty=0.5, reps=2, maxiter=150, seed=SEED
)

plot_optimization_history(history_qaoa_reg, title='QAOA Thresholding Optimization')
plt.show()

m = compute_metrics(y_sub, labels_qaoa_reg)
results['QAOA Regression (subsample)'] = m
print('QAOA Regression:', {k: f'{v:.3f}' for k, v in m.items()})

## 6. Quantum Distance k-NN

Quantum distance estimation uses the state fidelity between encoded data points
as a distance metric: d(x_1, x_2) = sqrt(1 - |<phi(x_1)|phi(x_2)>|^2).
Points with large average distance to their k nearest neighbors are flagged as anomalies.

In [ ]:
from quantum_anomaly_detection.circuits.feature_maps import build_angle_encoding_map
from quantum_anomaly_detection.methods.quantum_distance import detect_anomalies_knn

# Build feature map
fm_angle = build_angle_encoding_map(N_QUBITS)
fm_angle.draw('mpl', fold=20, scale=CIRCUIT_SCALE)

In [ ]:
# Run on subset (distance matrix is O(n^2))
n_dist = min(80, len(X_test))
idx_dist = rng.choice(len(X_test), n_dist, replace=False)

preds_qdist, scores_qdist = detect_anomalies_knn(
    X_test[idx_dist], fm_angle, k=5, contamination=CONTAMINATION
)

m = compute_metrics(y_test[idx_dist], preds_qdist, scores_qdist)
results['Quantum Distance k-NN'] = m
print('Quantum Distance k-NN:', {k: f'{v:.3f}' for k, v in m.items()})

## 7. Classical Benchmarks

### Level A: Classical Anomaly Detection Methods

In [ ]:
from quantum_anomaly_detection.classical.benchmarks import (
    run_isolation_forest, run_svm, run_lof, run_elliptic_envelope
)

# Isolation Forest
preds_if, scores_if = run_isolation_forest(X_train_normal, X_test, CONTAMINATION, SEED)
results['Isolation Forest'] = compute_metrics(y_test, preds_if, scores_if)

# One-Class SVM (RBF kernel)
preds_ocsvm, scores_ocsvm = run_svm(X_train_normal, X_test, nu=CONTAMINATION)
results['One-Class SVM (RBF)'] = compute_metrics(y_test, preds_ocsvm, scores_ocsvm)

# Local Outlier Factor
preds_lof, scores_lof = run_lof(X_train_normal, X_test, contamination=CONTAMINATION)
results['LOF'] = compute_metrics(y_test, preds_lof, scores_lof)

# Elliptic Envelope
preds_ee, scores_ee = run_elliptic_envelope(X_train_normal, X_test, CONTAMINATION)
results['Elliptic Envelope'] = compute_metrics(y_test, preds_ee, scores_ee)

print('Classical methods computed.')

### Level B: QAOA vs Classical Threshold

Compare the QAOA thresholding result against a simple percentile-based classical
threshold on the same subsample of residuals.

In [ ]:
# Classical threshold on same subsample
threshold_classical = np.percentile(res_sub, 100 * (1 - CONTAMINATION))
labels_classical_thresh = (res_sub > threshold_classical).astype(int)

m_ct = compute_metrics(y_sub, labels_classical_thresh)
results['Classical Threshold (subsample)'] = m_ct

print(f'QAOA labels:     {labels_qaoa_reg}')
print(f'Classical labels: {labels_classical_thresh}')
print(f'True labels:     {y_sub.astype(int)}')

## 8. Comparison

### Results Table

In [ ]:
from quantum_anomaly_detection.evaluation.metrics import format_results_table

df = format_results_table(results)
print(df.to_string())

In [ ]:
from quantum_anomaly_detection.visualization.plots import plot_roc_curves

# ROC curves for methods with continuous scores
roc_data = {}
if 'Quantum Kernel SVM' in results:
    roc_data['Quantum Kernel'] = (y_test[idx_test], scores_qk)
roc_data['VQC Autoencoder'] = (y_test, scores_ae)
roc_data['Quantum Distance'] = (y_test[idx_dist], scores_qdist)
roc_data['Isolation Forest'] = (y_test, scores_if)
roc_data['One-Class SVM (RBF)'] = (y_test, scores_ocsvm)
roc_data['LOF'] = (y_test, scores_lof)

plot_roc_curves(roc_data, title='ROC Curves — Time Series Anomaly Detection')
plt.show()

In [ ]:
from quantum_anomaly_detection.visualization.plots import plot_timeseries_anomalies

# Show detected anomalies from VQC Autoencoder
detected_idx = np.where(preds_ae == 1)[0]
plot_timeseries_anomalies(X_test_raw[:50], detected_idx[detected_idx < 50],
                          title='VQC Autoencoder Detected Anomalies')
plt.show()

## Discussion

- **Quantum Kernel One-Class SVM** captures non-linear similarity structure in the quantum
  feature space, which can differentiate anomalous windows from normal ones.
- **VQC Autoencoder** learns a compressed representation of normal signals; anomalies
  that deviate structurally produce higher reconstruction error.
- **QAOA Regression** solves the thresholding problem as quantum optimization, but
  is limited by qubit count to small subsamples.
- **Quantum Distance k-NN** provides an intuitive geometric interpretation but
  scales quadratically with sample size.
- Classical methods (particularly Isolation Forest and LOF) remain competitive on
  this synthetic dataset due to its relatively simple structure.